In [11]:
import yaml
import argparse
import logging
from types import SimpleNamespace
import os
from pathlib import Path
import csv
import sys
import math
import numpy as np
import pandas as pd
import time
import joblib
import json 
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
sys.path.append('D:\\PythonProjects\\GraspDataProcessing\\src')
import graspdataprocessing as gdp


In [12]:
def load_config(config_path):
    """加载YAML配置文件"""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return SimpleNamespace(**config)

def update_config(config_path, updates):
    """更新YAML配置文件
    
    Args:
        config_path: 配置文件路径
        updates: 要更新的键值对字典
    """
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # 更新配置值
    config.update(updates)
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)

def setup_logging(config):
    """配置日志系统"""
    os.makedirs("logs", exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler("logs/machine_learning_training.log"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

def setup_directories(root_path):
    """创建必要的目录结构"""
    root_path = Path(root_path)
    directories = ["models", "descripotors", "descripotors_stay", "test_data", "roc_curves", "results"]
    
    for directory in directories:
        (root_path / directory).mkdir(parents=True, exist_ok=True)
    
    return root_path

def initialize_results_file(result_csv_path, logger):
    """初始化结果CSV文件"""
    try:
        if not result_csv_path.exists():
            with result_csv_path.open(mode="w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow([
                    'training_time', 'eval_time', 'abinitio_time', 'all_time',
                    'f1', 'roc_auc', 'accuracy', 'precision', 'recall',
                    'Es', 'abimport_csfnum', 'MLimport_csfnum', 'MLsampling_ratio', 'next_itr_num',
                    'weight', 'f1_train', 'roc_auc_train', 'accuracy_train', 'precision_train', 'recall_train'
                ])
    except IOError as e:
        logger.error(f"无法创建结果文件 {result_csv_path}: {str(e)}")
        raise

def validate_initial_files(config, root_path, logger):
    """验证初始文件的存在和有效性"""
    # 验证初始CSFs文件
    initial_csfs_path = root_path / config.target_pool_file
    try:
        if not initial_csfs_path.is_file():
            logger.error(f"初始CSFs文件无效或不存在: {initial_csfs_path}")
            raise FileNotFoundError(f"初始CSFs文件无效或不存在: {initial_csfs_path}")
        logger.info(f"成功加载初始CSFs文件: {initial_csfs_path}")
    except PermissionError as e:
        logger.error(f"无权限访问CSFs文件: {initial_csfs_path}")
        raise
    except Exception as e:
        logger.error(f"加载CSFs文件时发生未知错误: {str(e)}")
        raise
    
    return initial_csfs_path

def check_configuration_coupling(config, energy_level_data_pd, logger):
    """检查组态耦合是否正确"""
    cal_configuration_set = set(energy_level_data_pd['configuration'])
    
    if set(config.spetral_term).issubset(cal_configuration_set):
        logger.info(f"cal_loop {config.cal_loop_num} 组态耦合正确")
        return True
    else:
        logger.error(f"cal_loop {config.cal_loop_num} 组态耦合错误")
        return False
    
def check_convergence(config, sum_num_list, logger):
    """检查收敛性"""
    # 这里需要Es_term的历史数据，暂时先跳过收敛检查
    # 在实际使用中，需要维护能量项的历史记录
    logger.info("收敛检查功能需要能量历史数据，当前跳过")
    return False

In [4]:
config_file = 'config.yaml'  # 配置文件的路径
config = load_config(config_file)

In [5]:
"""主程序逻辑"""
config.file_name = f'{config.conf}_{config.cal_loop_num}'
logger = setup_logging(config)
logger.info("机器学习训练程序启动")
execution_time = time.time()

# 设置目录结构
root_path = setup_directories(config.root_path)

# 初始化结果文件
result_csv_path = root_path / 'results/results.csv'
initialize_results_file(result_csv_path, logger)

# 验证初始文件
validate_initial_files(config, root_path, logger)

logger.info(f"初始比例: {config.initial_ratio}")
logger.info(f"光谱项: {config.spetral_term}")

2025-06-07 16:17:29,981 - INFO - 机器学习训练程序启动
2025-06-07 16:17:29,982 - INFO - 成功加载初始CSFs文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.c
2025-06-07 16:17:29,982 - INFO - 初始比例: 0.09
2025-06-07 16:17:29,982 - INFO - 光谱项: ['5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D', '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D']


In [6]:
target_pool_path = root_path.joinpath(config.conf)


In [8]:
ttttt = gdp.load_csfs_binary(target_pool_path)

In [7]:
asdggg = gdp.load_descriptors_with_multi_block(target_pool_path, file_format='npy')

Descriptors loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_data.npy
Labels loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_multi_block.npy


In [9]:
asdggg[1]

array([      0,       1,       2, ..., 4284443, 4284444, 4284445])

In [49]:
try:
    # 加载数据文件
    energy_level_data_pd, rmix_file_data, raw_csf_data, cal_csfs_data = load_data_files(config, root_path)
    
    # 检查组态耦合
    cal_result = check_configuration_coupling(config, energy_level_data_pd, logger)
    
    # 初始化索引
    indices_temp = list(range(raw_csf_data.CSFs_block_length[0]))
    sum_num_list = []

except Exception as e:
        logger.error(f"程序执行过程中发生错误: {str(e)}")
        raise

Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded
data file type: level data
energy levels data has been written into LEVEL/cv4odd1as3_odd4_1.level_level.csv csv file
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m loaded.
data file type: rmcdhf mix_coefficient_data
g92mix: G92MIX
 nblock = 1,       ncftot =   385600,          nw =  41,            nelec =   64


100%|██████████| 1/1 [00:00<00:00, 653.32it/s]

cycle jblock = 1
 Block no. = 1, 2J+1 = 9, Parity = -1, No. of eigenvalues = 2, No. of CSFs = 385600

    Energy levels for ...
Rydberg constant is  109737.31568508

---------------------------------------------
 No Pos  J  Parity   Energy Total    Levels
                      (a.u.)         (cm^-1)
---------------------------------------------

  1  1   4   +    -11274.7413433        0.00
  2  0   4   +    -11274.7204780     4579.40
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.c loaded.


Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c loaded.


2025-06-05 13:45:43,866 - INFO - cal_loop 1 组态耦合正确


In [15]:
if cal_result:
    # 记录能量信息
    for level in rmix_file_data.level_List:
        logger.info(f"迭代能量：{level}")
    logger.info("耦合正确")
    logger.info("************************************************")

2025-06-04 17:15:18,233 - INFO - 迭代能量：-11274.741343317686
2025-06-04 17:15:18,233 - INFO - 迭代能量：-11274.720478024547
2025-06-04 17:15:18,234 - INFO - 耦合正确
2025-06-04 17:15:18,234 - INFO - ************************************************


In [41]:
logger.info("开始增强特征提取")
cut_off_csfs_indices_dict = gdp.batch_asfs_mix_square_above_threshold(
        rmix_file_data, float(config.cutoff_value)
    )

2025-06-05 10:53:39,869 - INFO - 开始增强特征提取


In [16]:
test = gdp.load_csfs_binary('D:\\PythonProjects\\as3_odd4\\cv4odd1as3_odd4')

In [22]:
asdgxzv = gdp.batch_blocks_csfs_final_coupling_J_collection(test.CSFs_block_data, 2)

第1个block包含4284446个csf


In [32]:
target_pool_csfs_hash_file = target_pool_path.with_suffix('.pkl')

In [28]:
selected_csfs_file_path = root_path.joinpath(config.selected_csfs_file)

In [29]:
selected_csfs_load = gdp.GraspFileLoad.from_filepath(selected_csfs_file_path, file_type='CSF')

Data file D:\PythonProjects\as3_odd4\mJ-1-90chosenas3_odd4.c loaded.


In [30]:
selected_csfs_data = selected_csfs_load.data_file_process()

In [33]:
selected_csfs_indices_dict = gdp.maping_two_csfs_indices(
                selected_csfs_data.CSFs_block_data, 
                target_pool_csfs_hash_file
            )

In [34]:
selected_csfs_indices_dict

{0: [720050,
  1200405,
  277663,
  595864,
  1121514,
  1447689,
  198772,
  719993,
  1428692,
  1200381,
  80692,
  595785,
  277639,
  720022,
  1121480,
  61695,
  1501109,
  1200378,
  1447686,
  198738,
  595824,
  720019,
  277636,
  1200364,
  719990,
  1121476,
  1499808,
  1428688,
  277622,
  80689,
  1200402,
  595821,
  198734,
  1121458,
  595781,
  1200398,
  720047,
  42139,
  277660,
  198716,
  1121510,
  61691,
  277656,
  1200375,
  720017,
  1121504,
  1447683,
  595860,
  40838,
  198768,
  277633,
  198762,
  719987,
  1121473,
  1200373,
  595819,
  1428684,
  80686,
  198731,
  277631,
  595777,
  1200390,
  1121470,
  798518,
  797793,
  1249935,
  1250397,
  494949,
  720039,
  1057618,
  798257,
  1250231,
  798519,
  720043,
  61687,
  277648,
  1250398,
  198728,
  515528,
  327655,
  327193,
  1070708,
  797724,
  134876,
  1121493,
  1249890,
  1459730,
  595848,
  327489,
  1200394,
  721520,
  1459840,
  327656,
  1201345,
  1413469,
  147966,
  19875

In [48]:
def check_configuration_coupling(config, energy_level_data_pd, logger):
    """检查组态耦合是否正确"""
    cal_configuration_set = set(energy_level_data_pd['configuration'])
    
    if set(config.spetral_term).issubset(cal_configuration_set):
        logger.info(f"cal_loop {config.cal_loop_num} 组态耦合正确")
        return True
    else:
        logger.error(f"cal_loop {config.cal_loop_num} 组态耦合错误")
        return False

def load_data_files(config, root_path):
    """加载数据文件"""
    if gdp is None:
        raise ImportError("graspdataprocessing 模块未正确导入")
    
    cal_path = root_path / f'{config.conf}_{config.cal_loop_num}'
    
    # 加载能级文件
    energy_level_file_path = cal_path / f'{config.conf}_{config.cal_loop_num}.level'
    energy_level_file_load = gdp.LevelsEnergyData.from_filepath(str(energy_level_file_path), 'LEVEL')
    energy_level_data_pd = energy_level_file_load.energy_level_2_pd()
    
    # 加载rmix文件
    rmix_file_path = cal_path / f'{config.conf}_{config.cal_loop_num}.m'
    rmix_file_load = gdp.GraspFileLoad.from_filepath(str(rmix_file_path), 'mix')
    rmix_file_data = rmix_file_load.data_file_process()
    
    # 加载原始CSF文件
    raw_csf_file_path = root_path / f'{config.target_pool_file}'
    raw_csf_file_load = gdp.GraspFileLoad.from_filepath(str(raw_csf_file_path), 'CSFs')
    raw_csf_data = raw_csf_file_load.data_file_process()
    
    cal_csfs_file_path = cal_path / f'{config.conf}_{config.cal_loop_num}.c'
    cal_csfs_file_laod = gdp.GraspFileLoad.from_filepath(str(cal_csfs_file_path), 'CSFs')
    cal_csfs_data = cal_csfs_file_laod.data_file_process()
    
    return energy_level_data_pd, rmix_file_data, raw_csf_data, cal_csfs_data

def load_total_csfs_once(config, logger):
    """一次性加载总CSFs文件并缓存，避免重复读取"""
    if gdp is None:
        raise ImportError("graspdataprocessing 模块未正确导入")
    
    cache_file = Path(f"cache_total_csfs_{config.conf}.pkl")
    
    # 检查缓存是否存在
    if cache_file.exists():
        logger.info("从缓存加载总CSFs文件")
        import pickle
        with cache_file.open('rb') as f:
            return pickle.load(f)
    
    # 首次加载并缓存
    logger.info("首次加载总CSFs文件并创建缓存")
    raw_csf_file_load = gdp.GraspFileLoad.from_filepath(str(config.target_pool_file), 'CSFs')
    raw_csf_data = raw_csf_file_load.data_file_process()
    
    # 缓存数据
    import pickle
    with cache_file.open('wb') as f:
        pickle.dump(raw_csf_data, f)
    
    return raw_csf_data

In [ ]:
def train_enhanced_model(features, config, logger):
    """使用增强特征训练模型"""
    if gdp is None:
        raise ImportError("graspdataprocessing 模块未正确导入")
    
    logger.info("开始训练增强模型")
    
    # 准备训练数据
    X = features['feature_vector'].reshape(1, -1)  # 全局特征
    y = features['labels']  # CSF重要性标签
    
    # 为了训练，我们需要创建样本
    # 使用重要CSF索引创建正样本，随机选择负样本
    important_indices = features['important_indices']
    
    # 创建平衡的训练集
    n_important = len(important_indices)
    n_total = len(y)
    
    # 选择所有重要CSF和等量的不重要CSF
    negative_indices = np.where(y == 0)[0]
    selected_negative = np.random.choice(
        negative_indices, 
        size=min(n_important * 2, len(negative_indices)), 
        replace=False
    )
    
    # 合并正负样本索引
    train_indices = np.concatenate([important_indices, selected_negative])
    X_train = np.tile(X, (len(train_indices), 1))  # 重复全局特征
    y_train = y[train_indices]
    
    # 划分训练测试集
    X_train, X_test, y_train, y_test = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    
    # 初始化或加载模型
    models_dir = Path("models")
    models_dir.mkdir(exist_ok=True)
    
    if config.cal_loop_num == 1:
        model = RandomForestClassifier(
            class_weight={0:1, 1:3}, n_estimators=1000, verbose=True, n_jobs=-1
        )
    else:
        model_path = models_dir / f"{config.conf}_{config.cal_loop_num-1}.pkl"
        if model_path.exists():
            model = joblib.load(model_path)
        else:
            model = RandomForestClassifier(
                class_weight={0:1, 1:3}, n_estimators=1000, verbose=True, n_jobs=-1
            )
    
    # 设置权重
    weight = [1, max(1, 12 - 2*config.cal_loop_num)]
    logger.info(f"权重: {weight}")
    
    # 训练模型
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # 保存模型
    model_file = models_dir / f"{config.file_name}.pkl"
    joblib.dump(model, model_file)
    
    # 评估模型
    y_pred = model.predict(X_test)
    accuracy = np.mean(y_pred == y_test)
    logger.info(f"模型训练完成，测试准确率: {accuracy:.4f}")
    
    return model, training_time, weight